# Deep Dive: Classifier/Router Workflow

## Problem card

- **User/trigger:** a customer submits a freeform support ticket through a web form or email.
- **Inputs:** the raw ticket text only (no pre-existing category, no account context beyond what the customer wrote).
- **Output:** a routed, specialist-composed response -- a billing answer, a technical-bug status update, an account-access resolution, a logged feature request, or a clarifying question.
- **Success criteria:** every ticket lands in exactly one of five fixed categories, the right specialist fixture answers it, and an ambiguous ticket is never silently guessed into the wrong bucket.
- **Topology:** a **classifier/router workflow**. One LLM call makes exactly one classification decision; a LangGraph conditional edge keyed directly on that decision hands control to a fixed specialist node. The LLM does not act again after that point.
- **Safety boundary:** a low-confidence or ambiguous ticket must route to a clarifying question, never to a best-guess specialist.

Support-ticket triage is one of the most common shapes of production LLM
system there is: a small, fixed number of mutually exclusive categories,
each with its own downstream handling logic, and a single classification
decision sitting in front of all of it. It looks almost too simple to be
interesting after notebooks 01-04's loops and multi-step plans -- which is
exactly why it earns its own deep dive. Most of the engineering care here
goes into keeping the boundary between "the one thing the LLM decides" and
"everything Python decides afterward" completely clean, and into treating a
wrong classification as seriously as a crashed tool call, because a
misrouted ticket does not look like a failure to the caller.


In [1]:
import os
import re
import warnings
import logging
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

load_dotenv(find_dotenv(usecwd=True))
PROVIDER = os.getenv("PROVIDER", "openai").lower()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

import shared  # scorecard helpers (standardized single-agent scorecard)

scorecard = shared.ScorecardCallback()


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=key, callbacks=[scorecard])
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.3, api_key=key, callbacks=[scorecard])
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o-mini


## Architecture choice: why a classifier/router workflow fits

`README.md`'s terminology section names four topologies so far -- true
agent, agentic workflow, self-evaluating workflow, parallel/multi-agent
workflow. This notebook names a fifth, because none of the first four
describe what is happening here:

> **Classifier/router workflow:** a single LLM classification decision
> drives a fixed conditional-edge branch; the LLM does not act or
> re-decide after that point. Every subsequent step -- which fixture to
> query, how to compose the reply, whether to escalate -- is owned by
> deterministic Python code.

| Approach | Fit | Reason |
|---|---|---|
| True agent (ReAct, notebook 01) | Wrong shape | There is no evolving evidence trail to explore -- one ticket, one decision, done. |
| Agentic workflow (plan-execute, notebook 02) | Overkill | There is no multi-step process to plan; the "plan" is a single categorical choice. |
| Self-evaluating workflow (notebook 03) | Overkill | There is no single artifact worth a generate-critique-revise loop; the risk is misrouting, not a subtly wrong SQL query. |
| **Classifier/router** | **Strong** | Exactly one bounded decision (which of 5 categories) selects a fixed downstream path; everything after that is deterministic. |

The distinction that matters most for this notebook: in every prior
notebook, the LLM's decision *was* the control flow, made fresh at every
step. Here, the LLM's decision selects a control-flow branch *once*, and
then gets out of the way. That difference is not cosmetic -- it changes
what you test (classifier accuracy against a labeled set, not a trajectory
of tool calls), what you version (the category taxonomy and each
handler's fixture logic, separately), and what a wrong output looks like
(a confident answer from the wrong specialist, not a stalled loop or a
recursion-limit hit).


## Why single-agent, single-classification-call is right here

The categories are mutually exclusive by construction -- a ticket about a
duplicate charge is never simultaneously a technical bug -- and each
specialist handler is a simple, deterministic-shaped lookup against one
fixture. There is no case here for a multi-agent setup (say, one small
agent per category running independently and a coordinator picking the
best answer): that would introduce coordination overhead (whose answer
wins if two categories both claim the ticket?) to solve a problem that a
single categorical decision already solves cleanly. Multi-agent
coordination earns its cost when perspectives must stay independent or
work is genuinely parallelizable (notebook 04's boundary evidence) --
neither condition holds when exactly one category is supposed to be
true and the rest are supposed to be false.

The single classification call is also intentionally *not* a full ReAct
loop. A ReAct agent could, in principle, decide for itself which handler
to call as a tool -- but that would hand the LLM a decision that does not
need the flexibility of free-form tool selection, and would make an
ambiguous ticket harder to catch (a ReAct agent might "try" a tool rather
than cleanly refusing to guess). Constraining the LLM to one Literal-typed
field, and letting a plain conditional edge own the branch, is the
least-powerful mechanism that still solves the problem -- the same
selection principle notebooks 01-04 apply to their own topology choices.


## Required tools: four mock fixtures, not real infra

Same framing as 01-04: these are local, in-memory Python dicts that imitate
the shape of real backend systems (a billing ledger, an incident/status
board, an account/permissions store, a product backlog) without touching
any real infrastructure. They exist to prove the routing and handler logic
end to end, not to demonstrate production data integration.

Each ticket in this notebook embeds an account id (e.g. `ACC-1001`) the way
a real support ticket would carry an authenticated account context. Each
handler extracts that id with a small, deterministic regex -- not another
LLM call -- and looks it up in its own fixture. That extraction step is
part of "everything after the classification is deterministic code": no
model call is spent turning ticket text into an account id.


In [2]:
# Mock billing ledger -- consulted only by the billing handler.
INVOICES = {
    "ACC-1001": {
        "status": "duplicate_charge_pending_refund",
        "amount_due": 0.0,
        "last_charge": 49.00,
        "note": "Duplicate charge detected on 2026-07-15 for invoice INV-2201; refund eligible, not yet issued.",
    },
    "ACC-1002": {
        "status": "past_due_disputed",
        "amount_due": 89.00,
        "last_payment_date": "2026-07-03",
        "note": "Payment of $89.00 recorded 2026-07-03 but not yet reconciled against invoice INV-3391.",
    },
}

# Mock incident/service-status board -- consulted only by the technical-bug handler.
INCIDENTS = {
    "ACC-2001": {
        "service": "reporting-export",
        "status": "investigating",
        "eta": "2026-08-02 18:00 UTC",
        "summary": "Elevated 500 errors on report export since 09:10 UTC; root cause under investigation.",
    },
    "ACC-2002": {
        "service": "file-upload",
        "status": "monitoring",
        "eta": "resolved, monitoring for recurrence",
        "summary": "Intermittent upload failures traced to a degraded storage node; failover applied at 07:45 UTC.",
    },
}

# Mock account/permissions store -- consulted only by the account-access handler.
ACCOUNTS = {
    "ACC-3001": {
        "locked": True,
        "mfa_status": "delivery_delayed",
        "note": "MFA SMS delivery delayed for carrier T-Mobile; account unlocked after manual identity verification.",
    },
    "ACC-3002": {
        "role": "member",
        "admin_seats_available": 2,
        "note": "Workspace has 2 unused admin seats; requester already has permission to invite teammates as admin.",
    },
}

# Mock product backlog -- the feature-request handler appends to this, it never reads from a fixture.
BACKLOG: list[dict] = []

print(f"Fixtures ready: {len(INVOICES)} invoice records, {len(INCIDENTS)} incident records, "
      f"{len(ACCOUNTS)} account records, backlog starts empty.")


Fixtures ready: 2 invoice records, 2 incident records, 2 account records, backlog starts empty.


## Context engineering choices

The classifier and the handlers deliberately see different, minimal slices
of context -- not the full state, and not each other's fixtures:

1. **The classifier sees only the raw ticket text.** No account fixture,
   no prior tickets, no system context beyond the taxonomy definition
   itself. Classification is meant to be cheap and fast -- a single short
   call -- and giving it more context than the text being classified would
   only slow it down and risk it leaning on irrelevant detail instead of
   the words actually in the ticket.
2. **Each handler sees only its own fixture plus the ticket.** The billing
   handler never sees `INCIDENTS`; the technical-bug handler never sees
   `INVOICES`. This is the same context-isolation principle from
   `agent_context_engineering.ipynb`'s `input_schema` restriction, applied
   at the node level instead of the tool level: a node should only be
   handed the state it actually needs to do its job, both for cost and
   because an over-provisioned node is a node that can accidentally act on
   the wrong data.
3. **The account id is extracted deterministically, not re-classified.**
   Turning `"Account ACC-1001: ..."` into the lookup key `"ACC-1001"` is a
   regex, not a second LLM call -- the same "don't spend a model call where
   a script suffices" principle notebook 03's deterministic precheck
   applies before its critique step.

The net effect: the only thing that crosses from the classifier into the
rest of the graph is one Literal-typed field. Everything else a handler
needs, it re-derives locally from the raw ticket and its own fixture.


## Other design considerations: the ambiguous ticket, and why misrouting is worse than it looks

A classifier must have a fifth option that is not really a category: `unclear`.
Some tickets genuinely do not contain enough signal to classify safely --
"things aren't working, please fix it" could be a technical bug, a billing
complaint about a feature they lost access to, or an account lockout.
When the classifier is not confident the ticket clearly belongs to one of
the four real categories, the right behavior is to ask a clarifying
question, never to force it into the closest-sounding bucket.

This matters more than it looks like it should, for a reason worth stating
plainly: **a wrong-but-confident classification is a more insidious
failure than an obvious agent error.** When notebook 01's ReAct agent hits
its step budget, or notebook 03's critique loop fails closed, the system
visibly admits it did not reach a trustworthy answer. A misrouted ticket
does the opposite -- it produces a perfectly fluent, plausible-sounding
answer from the *wrong* specialist. A billing question routed to the
technical-bug handler will still get a confident-sounding reply about
service status, because the handler has no way of knowing the ticket
was misrouted; it just answers the ticket it was handed. Nothing in the
output signals that anything went wrong. The customer may not notice
until the "resolution" doesn't actually address their problem, and by
then the failure is much harder to trace back to a single bad
classification than a crashed tool call or a recursion-limit hit would
have been.

Two practical consequences follow from this:

- **The `unclear` branch is not a last resort bolted on for completeness --
  it is a first-class category** that should be exercised in eval just as
  deliberately as the four real ones, with genuinely ambiguous tickets
  written specifically to trigger it.
- **Classifier accuracy against a labeled set is the primary metric for
  this topology**, in the same way that a critic's calibration (not just
  pipeline outcome) was the key eval dimension in notebook 03 -- a
  router that never says `unclear` should be viewed with the same
  suspicion as a critic that never rejects anything.


## The classifier and router, implemented

`TicketClassification` carries exactly one field the router needs:
`category`. Even though it is flat (no nested list, unlike the
`Critique.issues` bug hit in notebook 03), this notebook still uses
`method="json_schema"` for the structured-output call -- cheap insurance
given that pattern has already broken this series' default tool-calling
extraction twice, in two different notebooks, for two different reasons.


In [3]:
from typing import Literal, Optional, TypedDict
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

TicketCategory = Literal["billing", "technical_bug", "account_access", "feature_request", "unclear"]

ACCOUNT_ID_PATTERN = re.compile(r"ACC-\d+")


class TicketClassification(BaseModel):
    category: TicketCategory = Field(
        description=(
            "Exactly one category. Use 'unclear' whenever the ticket does not contain enough "
            "concrete signal to confidently pick one of the other four -- never guess."
        )
    )


class TriageState(TypedDict):
    ticket_text: str
    account_id: Optional[str]
    category: Optional[TicketCategory]
    response: Optional[str]


# method="json_schema": the same defensive choice notebook 03 adopted after a real
# nested-list structured-output bug. This field is flat, but staying consistent
# with the safer extraction path costs nothing for a single categorical field.
classifier_llm = llm.with_structured_output(TicketClassification, method="json_schema")


def extract_account_id(ticket_text: str) -> Optional[str]:
    # Deterministic extraction, not a second LLM call -- see the context-engineering
    # section above for why this stays a regex rather than another classification.
    match = ACCOUNT_ID_PATTERN.search(ticket_text)
    return match.group(0) if match else None


def classify_node(state: TriageState) -> dict:
    # The classifier sees ONLY the raw ticket text -- no fixtures, no prior state.
    classification = classifier_llm.invoke(
        "Classify this support ticket into exactly one category: billing, technical_bug, "
        "account_access, feature_request, or unclear. Use 'unclear' if the ticket does not "
        "give you enough concrete signal to confidently choose one of the other four -- do "
        "not guess.\n\nTicket:\n" + state["ticket_text"]
    )
    print(f"[classify_node] category={classification.category}")
    return {"category": classification.category, "account_id": extract_account_id(state["ticket_text"])}


def route_by_category(state: TriageState) -> str:
    # The router is a plain conditional edge keyed directly on the structured field --
    # no keyword matching, no second model call. This is the entire "routing decision."
    return state["category"]


def billing_handler(state: TriageState) -> dict:
    record = INVOICES.get(state["account_id"])
    if record is None:
        return {"response": f"Billing team notified for account {state['account_id']!r}; no invoice record found yet, escalating for manual lookup."}
    return {"response": (
        f"Billing update for {state['account_id']}: {record['note']} "
        f"Current amount due: ${record['amount_due']:.2f}."
    )}


def technical_bug_handler(state: TriageState) -> dict:
    record = INCIDENTS.get(state["account_id"])
    if record is None:
        return {"response": f"No active incident found for account {state['account_id']!r}; opening a new investigation ticket."}
    return {"response": (
        f"Status update for {record['service']} ({record['status']}): {record['summary']} "
        f"ETA: {record['eta']}."
    )}


def account_access_handler(state: TriageState) -> dict:
    record = ACCOUNTS.get(state["account_id"])
    if record is None:
        return {"response": f"No account record found for {state['account_id']!r}; routing to identity verification."}
    return {"response": f"Account update for {state['account_id']}: {record['note']}"}


def feature_request_handler(state: TriageState) -> dict:
    # No lookup -- this handler only logs, it never reads a fixture.
    entry = {"account_id": state["account_id"], "request": state["ticket_text"], "logged_at": "2026-08-01"}
    BACKLOG.append(entry)
    return {"response": (
        f"Thanks -- this has been logged to the product backlog (now {len(BACKLOG)} item(s)) "
        f"for the team to prioritize. We do not have an ETA to share yet."
    )}


def unclear_handler(state: TriageState) -> dict:
    # Does NOT guess a specialist. Asks a clarifying question instead.
    return {"response": (
        "Thanks for reaching out -- to route this to the right team, could you share a bit more "
        "detail? For example: is this about a charge or invoice, something not working as expected, "
        "trouble logging in or account permissions, or a feature you would like to see added?"
    )}


triage_builder = StateGraph(TriageState)
triage_builder.add_node("classify", classify_node)
triage_builder.add_node("billing", billing_handler)
triage_builder.add_node("technical_bug", technical_bug_handler)
triage_builder.add_node("account_access", account_access_handler)
triage_builder.add_node("feature_request", feature_request_handler)
triage_builder.add_node("unclear", unclear_handler)
triage_builder.add_edge(START, "classify")
triage_builder.add_conditional_edges("classify", route_by_category, {
    "billing": "billing",
    "technical_bug": "technical_bug",
    "account_access": "account_access",
    "feature_request": "feature_request",
    "unclear": "unclear",
})
triage_builder.add_edge("billing", END)
triage_builder.add_edge("technical_bug", END)
triage_builder.add_edge("account_access", END)
triage_builder.add_edge("feature_request", END)
triage_builder.add_edge("unclear", END)

triage_agent = triage_builder.compile()
print("Ticket-triage classifier/router compiled: 1 classify node, 5 fixed branches, all converging to END.")


Ticket-triage classifier/router compiled: 1 classify node, 5 fixed branches, all converging to END.


## Graph topology

Five fixed branches out of one classification decision, all converging to
`END`. Nothing loops back to `classify` -- unlike notebook 03's
critique/revise cycle, there is no revision step here, because there is
nothing to revise: a ticket that comes back `unclear` gets a clarifying
question, not another classification attempt on the same text.

```mermaid
graph TD
    START([START]) --> CLASSIFY[classify:<br/>one structured LLM call]
    CLASSIFY -->|category=billing| BILLING[billing_handler:<br/>reads INVOICES]
    CLASSIFY -->|category=technical_bug| BUG[technical_bug_handler:<br/>reads INCIDENTS]
    CLASSIFY -->|category=account_access| ACCESS[account_access_handler:<br/>reads ACCOUNTS]
    CLASSIFY -->|category=feature_request| FEATURE[feature_request_handler:<br/>appends to BACKLOG]
    CLASSIFY -->|category=unclear| UNCLEAR[unclear_handler:<br/>asks a clarifying question]
    BILLING --> END([END])
    BUG --> END
    ACCESS --> END
    FEATURE --> END
    UNCLEAR --> END
```


## Eval strategy: a real scripted eval against labeled tickets

Because the classifier's job is a single categorical decision, the primary
eval dimension is exactly what you would expect: does the structured
`category` field match a hand-labeled ground truth, across all five
categories including at least one genuinely ambiguous ticket that should
land on `unclear`. This is scripted, not an LLM-judge eval -- comparing a
Literal string to an expected string needs no model call at all.


In [4]:
# Hand-labeled ground truth. Covers all 5 categories, including one ticket
# (the last one) that is deliberately ambiguous -- it should classify as
# "unclear" rather than being forced into one of the other four.
GROUND_TRUTH = [
    {
        "ticket_text": "Account ACC-1001: I was charged twice for my subscription this month, can you refund the duplicate charge?",
        "expected_category": "billing",
    },
    {
        "ticket_text": "Account ACC-1002: My invoice shows a past due balance but I already paid on the 3rd, please check.",
        "expected_category": "billing",
    },
    {
        "ticket_text": "Account ACC-2001: The dashboard has been throwing a 500 error every time I try to export a report.",
        "expected_category": "technical_bug",
    },
    {
        "ticket_text": "Account ACC-2002: File uploads have been failing intermittently since this morning, is this a known issue?",
        "expected_category": "technical_bug",
    },
    {
        "ticket_text": "Account ACC-3001: I'm locked out of my account after resetting my password, and my MFA code isn't arriving.",
        "expected_category": "account_access",
    },
    {
        "ticket_text": "Account ACC-3002: Can you add my colleague as an admin on our team workspace?",
        "expected_category": "account_access",
    },
    {
        "ticket_text": "Account ACC-4001: It would be great if we could export reports directly to Google Sheets instead of CSV.",
        "expected_category": "feature_request",
    },
    {
        "ticket_text": "Account ACC-5001: Things aren't working right, please fix it.",
        "expected_category": "unclear",
    },
]

print(f"{len(GROUND_TRUTH)} labeled tickets, categories represented: "
      f"{sorted(set(item['expected_category'] for item in GROUND_TRUTH))}")


8 labeled tickets, categories represented: ['account_access', 'billing', 'feature_request', 'technical_bug', 'unclear']


In [5]:
from shared import invoke_with_budget

# invoke_with_budget is reused for consistency with the rest of this series'
# defensive-invoke convention, even though this graph has no cycles and a
# GraphRecursionError is not a realistic failure mode here (one classify
# node, one fixed branch, straight to END). The fallback simply reuses the
# unclear-branch response shape if it were ever somehow triggered.
def on_budget_fallback() -> dict:
    return {"category": "unclear", "response": "Unable to classify within the graph budget; routing to a clarifying question."}


def run_ticket(ticket_text: str) -> dict:
    return invoke_with_budget(
        triage_agent,
        {"ticket_text": ticket_text, "account_id": None, "category": None, "response": None},
        {},
        on_budget_fallback,
    )


scorecard.reset()
with shared.Timer() as _scorecard_timer:
    eval_results = []
    for item in GROUND_TRUTH:
        result = run_ticket(item["ticket_text"])
        correct = result["category"] == item["expected_category"]
        eval_results.append({
            "ticket_text": item["ticket_text"],
            "expected": item["expected_category"],
            "actual": result["category"],
            "correct": correct,
            "response": result["response"],
        })
        tag = "OK" if correct else "MISMATCH"
        print(f"[{tag}] expected={item['expected_category']!r} actual={result['category']!r}")

    accuracy = sum(r["correct"] for r in eval_results) / len(eval_results)
    print(f"\nClassifier accuracy: {accuracy:.0%} ({sum(r['correct'] for r in eval_results)}/{len(eval_results)})")
    if accuracy != 1.0:
        print("NOTE: not every labeled ticket classified correctly on this real run -- read the "
              "[MISMATCH] line(s) above and the scorecard's Boundary violations row below rather "
              "than treating this as a hard failure; this notebook no longer hard-asserts 100% "
              "so a real miss is visible instead of crashing the run.")

scorecard_call_count = scorecard.call_count
scorecard_elapsed_s = _scorecard_timer.elapsed_s
print(f"\n[Scorecard capture] {scorecard_call_count} real LLM calls, {scorecard_elapsed_s:.2f}s wall-clock across {len(GROUND_TRUTH)} tickets")


[classify_node] category=billing
[OK] expected='billing' actual='billing'


[classify_node] category=billing
[OK] expected='billing' actual='billing'


[classify_node] category=technical_bug
[OK] expected='technical_bug' actual='technical_bug'


[classify_node] category=technical_bug
[OK] expected='technical_bug' actual='technical_bug'


[classify_node] category=account_access
[OK] expected='account_access' actual='account_access'


[classify_node] category=feature_request
[MISMATCH] expected='account_access' actual='feature_request'


[classify_node] category=feature_request
[OK] expected='feature_request' actual='feature_request'


[classify_node] category=unclear
[OK] expected='unclear' actual='unclear'

Classifier accuracy: 88% (7/8)
NOTE: not every labeled ticket classified correctly on this real run -- read the [MISMATCH] line(s) above and the scorecard's Boundary violations row below rather than treating this as a hard failure; this notebook no longer hard-asserts 100% so a real miss is visible instead of crashing the run.

[Scorecard capture] 8 real LLM calls, 6.87s wall-clock across 8 tickets


## Standardized single-agent scorecard

The same operational-metrics vocabulary as every other notebook in this
series (`shared.print_scorecard`), captured from this notebook's own real
run above (all 8 labeled tickets), not recomputed separately.

In [6]:
_mismatches = sum(1 for r in eval_results if not r["correct"])
_model_name = OPENAI_MODEL if PROVIDER == "openai" else ANTHROPIC_MODEL
_cost_estimate = shared.estimate_cost_usd(scorecard_call_count, _model_name)

shared.print_scorecard([
    ("Outcome correctness", f"{accuracy:.0%} ({len(GROUND_TRUTH) - _mismatches}/{len(GROUND_TRUTH)})", "Classified category matches hand-labeled ground truth, across all 5 categories"),
    ("Model-call count", str(scorecard_call_count), "Real LLM calls across all 8 tickets (1 classify call per ticket -- no re-decision after routing)"),
    ("Latency", f"{scorecard_elapsed_s:.2f}s", "Wall-clock for all 8 tickets"),
    ("Estimated cost", f"${_cost_estimate:.4f}", f"shared.estimate_cost_usd({scorecard_call_count} calls, {_model_name}) -- illustrative, not an exact invoice"),
    ("Replans/retries", "N/A", "No revision step -- an unclear ticket routes to a clarifying question, not another classification attempt"),
    ("Tool calls", "N/A", "Handlers use deterministic Python fixture lookups (INVOICES/INCIDENTS/ACCOUNTS/BACKLOG), not LLM-bound tools"),
    ("Human interventions", "N/A", "No HITL gate in this notebook"),
    ("Boundary violations", f"{_mismatches} misrouted ticket(s)", "A ticket landing in the wrong specialist branch -- structurally possible here (unlike wrong_domain_call_rate elsewhere), since routing correctness depends entirely on classifier accuracy, not a tool-binding guarantee"),
    ("Context-size proxy", "1 ticket, 0 tools/call", "The classifier sees only the raw ticket text -- no fixtures, no prior tickets, no tool bindings"),
])


Metric                  Value                 Why it matters
------------------------------------------------------------------------------------------------
Outcome correctness     88% (7/8)             Classified category matches hand-labeled ground truth, across all 5 categories
Model-call count        8                     Real LLM calls across all 8 tickets (1 classify call per ticket -- no re-decision after routing)
Latency                 6.87s                 Wall-clock for all 8 tickets
Estimated cost          $0.0014               shared.estimate_cost_usd(8 calls, gpt-4o-mini) -- illustrative, not an exact invoice
Replans/retries         N/A                   No revision step -- an unclear ticket routes to a clarifying question, not another classification attempt
Tool calls              N/A                   Handlers use deterministic Python fixture lookups (INVOICES/INCIDENTS/ACCOUNTS/BACKLOG), not LLM-bound tools
Human interventions     N/A                   No HITL gate in

**Expected output, and a real miss on this run worth reading precisely
rather than assuming a clean pass**: seven of eight labeled tickets
classified correctly, including `ACC-5001`'s deliberately vague ticket
landing on `unclear` rather than being forced into `technical_bug` or
`account_access` just because those are plausible guesses. **One ticket
misrouted on this run**: `ACC-3002` ("Can you add my colleague as an
admin on our team workspace?"), labeled `account_access` (an
access-group/permission-membership request for a named user), came back
classified as `feature_request` instead. Read why this one is genuinely
ambiguous rather than a nonsense error: "add \[someone\] as an admin" can
plausibly read either as a permissions/access action on an existing
workspace (the intended reading) or as a request for a capability the
product doesn't yet support (which is a defensible misreading if the
classifier weighted "add ... as" more like a feature ask than an
access-control action). This is a labeled-data eval failure in the same
spirit as notebook 01's test-isolation bug -- worth tracing to root
cause (should the taxonomy's `account_access` definition explicitly
name "granting/adding permissions for a teammate" as in-category?) rather
than patched by hand-tuning this one example. If the classifier ever
mis-routes one of the four concrete categories, that is exactly the kind
of signal this eval set is designed to surface.

### Eval strategy, generalized

| Dimension | What it catches | Scripted or LLM judge? |
|---|---|---|
| Classification accuracy vs. ground truth | The router sending a ticket to the wrong specialist | Scripted -- compare the structured `category` field to a labeled set |
| Unclear-branch recall | A classifier that never admits ambiguity, always forcing a guess | Scripted -- the ground-truth set must include genuinely ambiguous tickets |
| Handler correctness (independent of classification) | A handler that mishandles a correctly-routed ticket (e.g. a bad fixture lookup) | Scripted -- feed each handler a known account id directly and check its response references the right fixture fields |
| Response quality within a category | Whether the composed reply is actually helpful, not just correctly routed | Would need an LLM judge; out of scope for this notebook's scripted eval |

## Other design considerations, continued

- **Add a category only when a fixture and handler exist for it.** It is
  tempting to let the classifier's taxonomy grow ad hoc; every category
  the classifier can output must have a real downstream branch, or a
  correctly-classified ticket has nowhere deterministic to go.
- **Treat `unclear` as measured behavior, not a fallback you hope is rare.**
  Track its rate over time in production the way you would track a step
  budget being hit -- a rising `unclear` rate is a signal the taxonomy or
  prompt needs attention, not noise to suppress.
- **Keep the classifier prompt free of fixture data.** Nothing about
  `INVOICES`, `INCIDENTS`, or `ACCOUNTS` should leak into the classification
  call -- if the classifier starts reasoning about invoice amounts to decide
  a category, the context-isolation boundary from this notebook's
  context-engineering section has already been broken.
- **A misrouted ticket should be recoverable, not just measured.** In a real
  system, the specialist's own reply should include enough detail (e.g. "if
  this isn't about your invoice, let us know") that a human or the customer
  can catch a wrong route even when the eval set didn't.


## Revision summary

- A classifier/router workflow is a fifth topology alongside true agent,
  agentic workflow, self-evaluating workflow, and parallel/multi-agent: one
  LLM classification decision selects a fixed conditional-edge branch, and
  the LLM never acts or re-decides after that point.
- It fits when categories are mutually exclusive and each downstream branch
  is a simple, deterministic-shaped handler -- exactly the case where a
  multi-agent-per-category design would add coordination cost for no
  benefit.
- Context isolation applies at the node level: the classifier sees only raw
  ticket text; each handler sees only its own fixture, not the others'.
- `unclear` is a first-class branch, not an afterthought -- a classifier
  that never uses it should be treated with the same suspicion as a critic
  (notebook 03) that never rejects anything.
- A wrong-but-confident classification is more dangerous than an obvious
  agent failure, because the wrong specialist still produces a fluent,
  plausible-sounding answer with no visible signal that anything broke.
- Eval this topology the way you would eval any classifier: accuracy against
  a labeled set that deliberately includes ambiguous cases, plus independent
  checks on each handler's own correctness.

## Shared study guide

The common workflow-vs-agent explanation, interview framing, and glossary
are centralized in `00_architecture_landscape.ipynb`. Return there for the
shared vocabulary; this notebook keeps only topology-specific questions and
assignments.

## Checkpoint questions

1. **Q: Why isn't this notebook's topology just called "a ReAct agent with
   five tools"?**
   A: In ReAct, the LLM chooses which tool to call and can call several in
   sequence based on evolving evidence. Here, the LLM makes exactly one
   categorical decision, and a plain conditional edge -- not the LLM --
   determines the single downstream node. There is no tool selection loop.

2. **Q: Why does the classifier see only the raw ticket text, and not the
   account fixtures?**
   A: Keeping the classification call minimal keeps it cheap and fast, and
   prevents the classifier from reasoning about billing or incident details
   it has no business seeing before a category is even decided -- the same
   context-isolation principle as `input_schema` restriction in
   `agent_context_engineering.ipynb`, applied to a node instead of a tool.

3. **Q: Why is a wrong classification more dangerous than a ReAct agent
   hitting its step budget?**
   A: The step-budget case is visibly incomplete -- the system can say so.
   A misrouted ticket produces a fluent, complete-looking answer from the
   wrong specialist, with nothing in the output signaling anything went
   wrong; the failure only becomes visible when the customer notices their
   actual problem wasn't addressed.

## Learner assignments

1. **Add a sixth category.** Introduce `shipping_and_delivery` with its own
   fixture and handler, add at least two labeled tickets for it to
   `GROUND_TRUTH`, and confirm the eval still passes at 100%. Notice what
   has to change (the `Literal`, the conditional-edges mapping, a new node)
   and what does not (the classifier's prompt shape, the other handlers).

2. **Add a confidence field and a stricter unclear threshold.** Extend
   `TicketClassification` with a `confidence: Literal["low", "medium",
   "high"]` field, and route to `unclear` whenever confidence is `"low"`
   even if a concrete category was also returned. Add a test ticket that is
   category-obvious but written in a way that should still trigger low
   confidence (e.g. heavy sarcasm or contradictory statements), and confirm
   it correctly routes to `unclear`.

3. **Write the handler-independent eval.** Following the "handler
   correctness" row in the generalized eval table above, write a small test
   that calls `billing_handler`, `technical_bug_handler`, and
   `account_access_handler` directly with a hand-built `TriageState` (bypass
   `classify_node` entirely), and assert each response references the
   correct fixture fields for a known account id. This is the same
   "test the critic directly, not just the pipeline" discipline from
   notebook 03's calibration eval, applied to this notebook's handlers.
